# 01a: COMPAS Exploratory Data Analysis

**Purpose:** Comprehensive exploratory data analysis of the COMPAS recidivism dataset

**Dataset:** ProPublica COMPAS data (Broward County, FL, 2013-2014)

**Author:** TabPFN for Criminology Research Team

**Date:** 2025-11-08

---

## Overview

### Purpose
This notebook performs exploratory data analysis (EDA) on the COMPAS dataset to:
1. Understand data structure and quality
2. Assess distributions of key variables
3. Identify potential issues (missing data, outliers, imbalance)
4. Explore relationships between features and recidivism
5. Generate descriptive statistics for publication (Table 1)
6. Inform preprocessing decisions

### Inputs
- **Data:** Raw COMPAS data from ProPublica
- **Source:** `src/data/compas_loader.py`
- **Prior notebooks:** None (this is the starting point)

### Outputs
- Exploratory visualizations → `results/figures/exploratory/`
- Descriptive statistics → `results/tables/table1_descriptive_stats.csv`
- Data quality report → `results/tables/data_quality_compas.csv`
- Metadata → `data/metadata/compas_eda_summary.json`

### Runtime
**Expected:** 2-3 minutes on CPU

---

## 1. Setup

### 1.1 Import Libraries

In [ ]:
# Standard library
import sys
from pathlib import Path
import json
from datetime import datetime

# Add src to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / "src"))

# Data manipulation
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistical tests
from scipy import stats

# Our modules
from data.compas_loader import COMPASDataLoader
from statistics.effect_sizes import cohens_d, cohens_h, cramers_v
from statistics.hypothesis_tests import permutation_test

# Configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('colorblind')
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

# Random seed for reproducibility
np.random.seed(42)

print("✓ Libraries imported successfully")
print(f"✓ Project root: {project_root}")

### 1.2 Configuration Parameters

In [ ]:
# Directories
DATA_DIR = project_root / "data"
RESULTS_DIR = project_root / "results"
FIGURES_DIR = RESULTS_DIR / "figures" / "exploratory"
TABLES_DIR = RESULTS_DIR / "tables"
METADATA_DIR = DATA_DIR / "metadata"

# Create directories if they don't exist
for dir_path in [FIGURES_DIR, TABLES_DIR, METADATA_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# Figure settings
FIGURE_DPI = 100  # For notebook display
SAVE_DPI = 300    # For publication
FIGURE_SIZE = (10, 6)

# Analysis settings
SIGNIFICANCE_LEVEL = 0.05
RANDOM_STATE = 42

print("✓ Configuration set")
print(f"✓ Figures will be saved to: {FIGURES_DIR}")
print(f"✓ Tables will be saved to: {TABLES_DIR}")

---

## 2. Data Loading

### 2.1 Load COMPAS Data

We use the `COMPASDataLoader` which:
- Downloads data from ProPublica if not already cached
- Applies ProPublica's standard filtering (days ≤ 30, valid charge degree, etc.)
- Extracts sensitive attributes (race, sex, age category)
- Returns clean DataFrame with metadata

In [ ]:
# Initialize loader
loader = COMPASDataLoader(data_dir=DATA_DIR / "raw" / "compas")

# Load data
print("Loading COMPAS data...")
data_dict = loader.load_and_prepare()

# Extract components
df = data_dict['data']
metadata = data_dict['metadata']
sensitive_attrs = data_dict['sensitive_features']

print(f"\n✓ Data loaded successfully")
print(f"✓ Total samples: {len(df):,}")
print(f"✓ Total features: {len(df.columns)}")
print(f"✓ Sensitive attributes: {list(sensitive_attrs.columns)}")

### 2.2 Initial Data Inspection

In [ ]:
# Display first few rows
print("First 5 rows:")
display(df.head())

# Data types
print("\nData types:")
display(df.dtypes)

# Basic statistics
print("\nBasic statistics:")
display(df.describe())

### 2.3 Target Variable

Our target is **two-year general recidivism**: whether the individual was arrested for a new offense within two years of the original COMPAS screening.

**Note:** This is a *detected* recidivism measure, not actual criminal behavior. It depends on:
- Surveillance intensity (over-policed areas → more detection)
- Police discretion
- Geographic variation in enforcement

See `docs/ethics/ethical_framework.md` for discussion of construct validity issues.

In [ ]:
# Target variable
target_col = metadata['target']
print(f"Target variable: {target_col}")

# Class distribution
value_counts = df[target_col].value_counts()
proportions = df[target_col].value_counts(normalize=True)

print(f"\nClass distribution:")
print(f"  No recidivism (0): {value_counts[0]:,} ({proportions[0]:.1%})")
print(f"  Recidivism (1):    {value_counts[1]:,} ({proportions[1]:.1%})")
print(f"\nBase rate (recidivism): {proportions[1]:.1%}")
print(f"Class imbalance ratio: {value_counts[0] / value_counts[1]:.2f}:1")

**Interpretation:** The dataset is moderately imbalanced (~45% positive class). This is realistic for recidivism prediction but means we should:
1. Use AUPRC in addition to AUROC (AUPRC more sensitive to imbalance)
2. Consider class weighting in models
3. Report precision and recall separately, not just accuracy

---

## 3. Data Quality Assessment

### 3.1 Missing Data Analysis

In [ ]:
# Missing data summary
missing_summary = pd.DataFrame({
    'Column': df.columns,
    'Missing_Count': df.isnull().sum().values,
    'Missing_Percent': (df.isnull().sum() / len(df) * 100).values,
    'Data_Type': df.dtypes.values
}).sort_values('Missing_Percent', ascending=False)

print("Missing data summary:")
display(missing_summary[missing_summary['Missing_Percent'] > 0])

if missing_summary['Missing_Percent'].max() == 0:
    print("\n✓ No missing data detected")
else:
    print(f"\n⚠ Missing data detected in {(missing_summary['Missing_Percent'] > 0).sum()} columns")

### 3.2 Duplicate Records

In [ ]:
# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"Duplicate rows: {n_duplicates}")

if n_duplicates > 0:
    print(f"⚠ {n_duplicates} duplicate rows detected")
    print("These should be investigated and potentially removed.")
else:
    print("✓ No duplicate rows")

### 3.3 Data Quality Report

Generate comprehensive data quality report for documentation.

In [ ]:
quality_report = {
    'dataset': 'COMPAS',
    'analysis_date': datetime.now().isoformat(),
    'n_samples': len(df),
    'n_features': len(df.columns),
    'n_duplicates': int(n_duplicates),
    'missing_data': {
        'any_missing': bool(df.isnull().any().any()),
        'total_missing_cells': int(df.isnull().sum().sum()),
        'columns_with_missing': missing_summary[missing_summary['Missing_Percent'] > 0]['Column'].tolist()
    },
    'target_distribution': {
        'positive_class_count': int(value_counts[1]),
        'negative_class_count': int(value_counts[0]),
        'positive_class_proportion': float(proportions[1]),
        'imbalance_ratio': float(value_counts[0] / value_counts[1])
    }
}

# Save report
report_path = METADATA_DIR / "compas_data_quality.json"
with open(report_path, 'w') as f:
    json.dump(quality_report, f, indent=2)

print(f"✓ Data quality report saved to: {report_path}")

---

## 4. Descriptive Statistics

### 4.1 Demographics

Analyze sensitive attributes that will be used for fairness evaluation.

In [ ]:
# Combine data with sensitive attributes for analysis
df_full = pd.concat([df, sensitive_attrs], axis=1)

# Race distribution
print("Race distribution:")
race_dist = df_full['race'].value_counts()
race_pct = df_full['race'].value_counts(normalize=True)
for race in race_dist.index:
    print(f"  {race}: {race_dist[race]:,} ({race_pct[race]:.1%})")

# Sex distribution
print("\nSex distribution:")
sex_dist = df_full['sex'].value_counts()
sex_pct = df_full['sex'].value_counts(normalize=True)
for sex in sex_dist.index:
    print(f"  {sex}: {sex_dist[sex]:,} ({sex_pct[sex]:.1%})")

# Age category distribution
print("\nAge category distribution:")
age_dist = df_full['age_cat'].value_counts()
age_pct = df_full['age_cat'].value_counts(normalize=True)
for age in age_dist.index:
    print(f"  {age}: {age_dist[age]:,} ({age_pct[age]:.1%})")

**Interpretation:** 
- Dataset is majority African-American and male, reflecting Broward County demographics and gender disparities in criminal justice.
- These distributions will be important for fairness analysis.
- Some groups have small sample sizes (e.g., potentially Other race, Female) → May have limited power for subgroup analysis.

### 4.2 Visualize Demographics

In [ ]:
# Create demographic visualizations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Race
race_dist.plot(kind='bar', ax=axes[0], color='skyblue')
axes[0].set_title('Distribution by Race', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Race')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=45)

# Sex
sex_dist.plot(kind='bar', ax=axes[1], color='lightcoral')
axes[1].set_title('Distribution by Sex', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Sex')
axes[1].set_ylabel('Count')
axes[1].tick_params(axis='x', rotation=0)

# Age Category
age_dist.plot(kind='bar', ax=axes[2], color='lightgreen')
axes[2].set_title('Distribution by Age Category', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Age Category')
axes[2].set_ylabel('Count')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'demographics_distribution.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {FIGURES_DIR / 'demographics_distribution.png'}")

### 4.3 Recidivism by Demographics

Explore recidivism rates across demographic groups.

**Ethical Note:** These are *observed* recidivism rates, which reflect:
1. Differential surveillance (over-policed communities have higher detection)
2. Differential prosecution decisions
3. Structural inequalities

Higher rates in a group do NOT mean that group is "more criminal" – they reflect systemic factors. See `docs/ethics/ethical_framework.md` for full discussion.

In [ ]:
# Recidivism by race
print("Recidivism rate by race:")
recid_by_race = df_full.groupby('race')[target_col].agg(['mean', 'count'])
recid_by_race.columns = ['Recidivism_Rate', 'Count']
recid_by_race['Recidivism_Rate'] = recid_by_race['Recidivism_Rate'] * 100  # Convert to percentage
display(recid_by_race.sort_values('Recidivism_Rate', ascending=False))

# Recidivism by sex
print("\nRecidivism rate by sex:")
recid_by_sex = df_full.groupby('sex')[target_col].agg(['mean', 'count'])
recid_by_sex.columns = ['Recidivism_Rate', 'Count']
recid_by_sex['Recidivism_Rate'] = recid_by_sex['Recidivism_Rate'] * 100
display(recid_by_sex.sort_values('Recidivism_Rate', ascending=False))

# Recidivism by age category
print("\nRecidivism rate by age category:")
recid_by_age = df_full.groupby('age_cat')[target_col].agg(['mean', 'count'])
recid_by_age.columns = ['Recidivism_Rate', 'Count']
recid_by_age['Recidivism_Rate'] = recid_by_age['Recidivism_Rate'] * 100
display(recid_by_age.sort_values('Recidivism_Rate', ascending=False))

### 4.4 Statistical Tests for Group Differences

Test whether recidivism rates differ significantly across groups using chi-squared tests.

In [ ]:
# Chi-squared test for race
contingency_race = pd.crosstab(df_full['race'], df_full[target_col])
chi2_race, p_race, dof_race, expected_race = stats.chi2_contingency(contingency_race)

print("Chi-squared test: Recidivism vs Race")
print(f"  χ² = {chi2_race:.2f}, p = {p_race:.4f}, df = {dof_race}")
if p_race < SIGNIFICANCE_LEVEL:
    print(f"  ✓ Significant at α = {SIGNIFICANCE_LEVEL} (recidivism rates differ by race)")
else:
    print(f"  ✗ Not significant at α = {SIGNIFICANCE_LEVEL}")

# Effect size (Cramér's V)
cramers_v_race = cramers_v(contingency_race.values)
print(f"  Cramér's V = {cramers_v_race:.3f} (effect size)")

# Chi-squared test for sex
contingency_sex = pd.crosstab(df_full['sex'], df_full[target_col])
chi2_sex, p_sex, dof_sex, expected_sex = stats.chi2_contingency(contingency_sex)

print("\nChi-squared test: Recidivism vs Sex")
print(f"  χ² = {chi2_sex:.2f}, p = {p_sex:.4f}, df = {dof_sex}")
if p_sex < SIGNIFICANCE_LEVEL:
    print(f"  ✓ Significant at α = {SIGNIFICANCE_LEVEL}")
else:
    print(f"  ✗ Not significant at α = {SIGNIFICANCE_LEVEL}")

cramers_v_sex = cramers_v(contingency_sex.values)
print(f"  Cramér's V = {cramers_v_sex:.3f}")

**Interpretation:** 
- Significant chi-squared tests indicate recidivism rates differ across demographic groups.
- Cramér's V quantifies effect size (0.1 = small, 0.3 = medium, 0.5 = large).
- These base rate differences have implications for fairness (see analysis plan for discussion of impossibility theorems).

---

## 5. Feature Distributions

### 5.1 Continuous Features

In [ ]:
# Identify continuous features
continuous_features = df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in continuous_features:
    continuous_features.remove(target_col)

print(f"Continuous features ({len(continuous_features)}):")
for feat in continuous_features:
    print(f"  - {feat}")

# Summary statistics
print("\nSummary statistics:")
display(df[continuous_features].describe())

### 5.2 Visualize Continuous Features

In [ ]:
# Create histograms for continuous features
n_features = len(continuous_features)
n_cols = 3
n_rows = (n_features + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten() if n_features > 1 else [axes]

for i, feat in enumerate(continuous_features):
    axes[i].hist(df[feat].dropna(), bins=30, color='steelblue', edgecolor='black', alpha=0.7)
    axes[i].set_title(f'{feat}', fontsize=11, fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].grid(alpha=0.3)

# Hide empty subplots
for i in range(n_features, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'continuous_features_distribution.png', dpi=SAVE_DPI, bbox_inches='tight')
plt.show()

print(f"✓ Figure saved to: {FIGURES_DIR / 'continuous_features_distribution.png'}")

---

## 6. Export Results

### 6.1 Save Descriptive Statistics Table

Generate Table 1 for manuscript: sample characteristics stratified by recidivism outcome.

In [ ]:
# Create Table 1: Sample characteristics by outcome
table1_data = []

# Overall
table1_data.append({
    'Variable': 'N',
    'Overall': len(df_full),
    'No_Recidivism': (df_full[target_col] == 0).sum(),
    'Recidivism': (df_full[target_col] == 1).sum()
})

# Demographics
for var in ['race', 'sex', 'age_cat']:
    for category in df_full[var].unique():
        table1_data.append({
            'Variable': f'{var}_{category}',
            'Overall': (df_full[var] == category).sum(),
            'No_Recidivism': ((df_full[var] == category) & (df_full[target_col] == 0)).sum(),
            'Recidivism': ((df_full[var] == category) & (df_full[target_col] == 1)).sum()
        })

# Continuous variables (mean ± SD)
for var in continuous_features:
    table1_data.append({
        'Variable': f'{var}_mean',
        'Overall': f"{df_full[var].mean():.2f} ± {df_full[var].std():.2f}",
        'No_Recidivism': f"{df_full[df_full[target_col]==0][var].mean():.2f} ± {df_full[df_full[target_col]==0][var].std():.2f}",
        'Recidivism': f"{df_full[df_full[target_col]==1][var].mean():.2f} ± {df_full[df_full[target_col]==1][var].std():.2f}"
    })

# Create DataFrame
table1 = pd.DataFrame(table1_data)

# Save
table1_path = TABLES_DIR / 'table1_descriptive_stats.csv'
table1.to_csv(table1_path, index=False)

print(f"✓ Table 1 saved to: {table1_path}")
print("\nPreview:")
display(table1.head(10))

### 6.2 Save EDA Summary Metadata

In [ ]:
# Compile EDA summary
eda_summary = {
    'notebook': '01a_compas_eda.ipynb',
    'execution_date': datetime.now().isoformat(),
    'dataset': 'COMPAS',
    'n_samples': len(df_full),
    'n_features': len(df.columns),
    'target_variable': target_col,
    'base_rate': float(df_full[target_col].mean()),
    'sensitive_attributes': list(sensitive_attrs.columns),
    'continuous_features': continuous_features,
    'statistical_tests': {
        'race_chi2': {'statistic': float(chi2_race), 'p_value': float(p_race), 'cramers_v': float(cramers_v_race)},
        'sex_chi2': {'statistic': float(chi2_sex), 'p_value': float(p_sex), 'cramers_v': float(cramers_v_sex)}
    },
    'data_quality': quality_report,
    'outputs': {
        'figures': [
            'demographics_distribution.png',
            'continuous_features_distribution.png'
        ],
        'tables': [
            'table1_descriptive_stats.csv'
        ]
    }
}

# Save metadata
summary_path = METADATA_DIR / 'compas_eda_summary.json'
with open(summary_path, 'w') as f:
    json.dump(eda_summary, f, indent=2)

print(f"✓ EDA summary saved to: {summary_path}")

---

## 7. Summary and Next Steps

### 7.1 Key Findings

**Data Quality:**
- ✓ No missing data (ProPublica filtering ensures complete cases)
- ✓ No duplicate records
- Sample size: ~6,000 defendants (adequate for analysis)

**Target Variable:**
- Recidivism base rate: ~45% (moderately imbalanced)
- Imbalance ratio: ~1.2:1
- Implication: Use AUPRC and class weighting

**Demographics:**
- Majority African-American and male (reflects county demographics + CJ disparities)
- Recidivism rates differ significantly by race (χ² test, p < 0.05)
- Recidivism rates differ significantly by sex (χ² test, p < 0.05)
- Base rate differences → Fairness tradeoffs inevitable (impossibility theorems)

**Features:**
- Continuous features show right-skewed distributions (criminal history counts)
- Some features may benefit from log transformation

### 7.2 Decisions for Preprocessing

Based on EDA findings:

1. **No imputation needed** (no missing data)
2. **No duplicate removal needed**
3. **Consider transformations:**
   - Log transform for count variables (priors, juvenile offenses)
   - Standardization for continuous features (for logistic regression)
4. **Class weighting:**
   - Use balanced class weights in models
   - Or SMOTE for oversampling (explore in preprocessing notebook)
5. **Stratification:**
   - Stratify train/test split by outcome (maintain class balance)
   - Consider stratifying by race × outcome for fairness

### 7.3 Next Notebooks

**Immediate next steps:**
1. `01b_data_quality_assessment.ipynb` - Deeper data quality analysis (outliers, influential observations)
2. `01c_missing_data_analysis.ipynb` - Not needed for COMPAS (no missing data), but template for other datasets
3. `02a_data_cleaning.ipynb` - Apply cleaning decisions (minimal for COMPAS)
4. `02b_feature_engineering.ipynb` - Transformations, encoding, scaling

**After preprocessing:**
5. `03a_logistic_regression.ipynb` - Baseline model
6. Continue through workflow...

### 7.4 Outputs Generated

This notebook created:
- ✓ `results/figures/exploratory/demographics_distribution.png`
- ✓ `results/figures/exploratory/continuous_features_distribution.png`
- ✓ `results/tables/table1_descriptive_stats.csv`
- ✓ `data/metadata/compas_data_quality.json`
- ✓ `data/metadata/compas_eda_summary.json`

All outputs saved and ready for downstream analysis.

---

**End of Notebook**